# High-Yield Nepali Manual Corpus Collector (Live News & Literary Portals)

**Fast Multi-Threaded Architecture (~15–20 minutes to collect 50M+ manual characters)**:
1. **Ratopati Archives**: 50,000+ live articles (`https://www.ratopati.com/story/{id}`).
2. **Setopati Archives**: 30,000+ live articles across Politics, Social, Opinion, Kinmel (`https://www.setopati.com/{category}/{id}`).
3. **OnlineKhabar & NepalNews**: Native Devanagari news and literature.
4. **Direct Google Drive Integration**: Flushes `manual_news_shard_XXXX.jsonl` to Drive, re-runs 90/5/5 splits, and updates total manual token share.

## 0. Setup & Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q trafilatura sentencepiece tabulate requests beautifulsoup4

import json, os, re, hashlib, gc, time, math, collections, random, requests, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import trafilatura
import sentencepiece as spm
from tabulate import tabulate

DRIVE_ROOT = "/content/drive/MyDrive/24KaratAu/nepali"
MANUAL_DIR = f"{DRIVE_ROOT}/cleaned/manual"
NONWEB_DIR = f"{DRIVE_ROOT}/cleaned/automated_nonweb"
WEB_DIR    = f"{DRIVE_ROOT}/cleaned/automated_web"
DRIVE_SPLITS_DIR = f"{DRIVE_ROOT}/splits"
DRIVE_TOKENIZER_DIR = f"{DRIVE_ROOT}/tokenizer"
MANIFEST_DIR = f"{DRIVE_ROOT}/manifests"

# Local fast staging directories (zero FUSE crashes)
LOCAL_SPLITS_DIR = "/content/splits"
LOCAL_TOKENIZER_DIR = "/content/tokenizer"

for d in [MANUAL_DIR, NONWEB_DIR, WEB_DIR, DRIVE_SPLITS_DIR, DRIVE_TOKENIZER_DIR, MANIFEST_DIR, LOCAL_SPLITS_DIR, LOCAL_TOKENIZER_DIR]:
    os.makedirs(d, exist_ok=True)

print("Setup complete. Connected to Google Drive with fast local staging.")

## 1. High-Yield Multi-Threaded Manual Crawler (Ratopati + Setopati + Portals)

In [ ]:
MIN_CHARS = 150
MIN_DEV_RATIO = 0.40
SHARD_SIZE = 5000

def devanagari_ratio(text):
    if not text: return 0.0
    return sum(1 for c in text if '\u0900' <= c <= '\u097F') / len(text)

def clean_text(text):
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

# 1. Generate Live Verified URL Lists
print('Generating verified live URL lists...')
target_urls = []

# Ratopati (IDs from 585,000 downwards)
target_urls.extend([f'https://www.ratopati.com/story/{i}' for i in range(585000, 535000, -1)])

# Setopati across core categories
for cat in ['politics', 'social', 'kinmel', 'opinion', 'sports', 'art']:
    target_urls.extend([f'https://www.setopati.com/{cat}/{i}' for i in range(396500, 385000, -1)])

print(f'Total Target Candidate URLs: {len(target_urls):,}')

# 2. Multi-Threaded Fetch & Extraction Engine
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

buffer_docs = []
shard_idx = len(list(Path(MANUAL_DIR).glob('manual_news_shard_*.jsonl')))
total_manual_chars = 0
total_manual_docs = 0

def flush_shard():
    global buffer_docs, shard_idx
    if not buffer_docs: return
    sf = f'{MANUAL_DIR}/manual_news_shard_{shard_idx:04d}.jsonl'
    with open(sf, 'w', encoding='utf-8') as f:
        for d in buffer_docs:
            f.write(json.dumps(d, ensure_ascii=False) + '\n')
    print(f'  Saved Shard to Drive: {Path(sf).name} ({len(buffer_docs):,} docs)')
    buffer_docs = []
    shard_idx += 1
    gc.collect()

def fetch_and_clean_url(url):
    try:
        r = requests.get(url, headers=headers, timeout=4)
        if r.status_code == 200:
            text = trafilatura.extract(r.text, include_tables=False)
            if text:
                clean_p = clean_text(text)
                if len(clean_p) >= MIN_CHARS and devanagari_ratio(clean_p) >= MIN_DEV_RATIO:
                    h = hashlib.sha256(clean_p.encode('utf-8')).hexdigest()
                    source_name = 'Ratopati' if 'ratopati' in url else 'Setopati'
                    return {
                        'doc_id': h[:16],
                        'url': url,
                        'source': source_name,
                        'source_type': 'news_web',
                        'collection_type': 'manual',
                        'language': 'nepali',
                        'text': clean_p,
                        'clean_text_characters': len(clean_p),
                        'devanagari_ratio': round(devanagari_ratio(clean_p), 4),
                        'text_sha256': h
                    }
    except: pass
    return None

# 3. Run parallel extraction with 30 concurrent workers
print('\nStarting Parallel Multi-Threaded Crawl (30 workers)...')
start_time = time.time()

with ThreadPoolExecutor(max_workers=30) as executor:
    futures = {executor.submit(fetch_and_clean_url, u): u for u in target_urls}
    processed = 0
    
    for future in as_completed(futures):
        processed += 1
        doc = future.result()
        if doc:
            buffer_docs.append(doc)
            total_manual_chars += doc['clean_text_characters']
            total_manual_docs += 1
            if len(buffer_docs) >= SHARD_SIZE:
                flush_shard()
                
        if processed % 2000 == 0:
            elapsed = time.time() - start_time
            rate = processed / max(1, elapsed)
            print(f'  Processed: {processed:,} / {len(target_urls):,} URLs ({rate:.1f} urls/s) | Accepted Docs: {total_manual_docs:,} | Chars: {total_manual_chars:,} (~{total_manual_chars//5:,} tokens)')

flush_shard()
print(f'\nCRAWL COMPLETE: Collected {total_manual_docs:,} new manual documents ({total_manual_chars:,} chars, ~{total_manual_chars//5:,} tokens).')

## 2. Re-run 90/5/5 Splits & Final Balanced Corpus Accounting

In [ ]:
random.seed(42)
# Write directly to local NVMe disk (/content/splits) to eliminate Google Drive FUSE socket crashes
out = {s: open(f"{LOCAL_SPLITS_DIR}/{s}.txt", "w", encoding="utf-8") for s in ["train", "val", "test"]}
jout = {s: open(f"{LOCAL_SPLITS_DIR}/{s}.jsonl", "w", encoding="utf-8") for s in ["train", "val", "test"]}
counts = {"train": 0, "val": 0, "test": 0}
manual_char_total, auto_char_total = 0, 0

print("Streaming all shards from Drive and building 90/5/5 splits in local fast storage...")
for d in [MANUAL_DIR, NONWEB_DIR, WEB_DIR]:
    if not os.path.exists(d): continue
    shards = sorted(Path(d).glob("*.jsonl"))
    print(f"  Processing directory: {d} ({len(shards)} shards)")
    for sf in shards:
        with open(sf, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip(): continue
                r = random.random()
                s = "train" if r < 0.90 else ("val" if r < 0.95 else "test")
                jout[s].write(line)
                doc = json.loads(line)
                t = doc.get("text", "")
                c = doc.get("clean_text_characters", len(t))
                if doc.get("collection_type") == "manual":
                    manual_char_total += c
                else:
                    auto_char_total += c
                if t.strip():
                    out[s].write(t.strip() + "

")
                counts[s] += 1

for f in out.values(): f.close()
for f in jout.values(): f.close()

tot = manual_char_total + auto_char_total
print("=" * 60)
print(f"FINAL BALANCED CORPUS SUMMARY")
print(f"  Manual Tokens:    ~{manual_char_total // 5:,} tokens ({manual_char_total / max(1, tot) * 100:.1f}%)")
print(f"  Automated Tokens: ~{auto_char_total // 5:,} tokens ({auto_char_total / max(1, tot) * 100:.1f}%)")
print(f"  Total Tokens:     ~{tot // 5:,} tokens")
for s in ["train", "val", "test"]: print(f"  {s}: {counts[s]:,} docs")
print("=" * 60)

# Copy completed split files in bulk to Google Drive
print("
Flushing verified splits to Google Drive...")
for s in ["train", "val", "test"]:
    shutil.copy(f"{LOCAL_SPLITS_DIR}/{s}.txt", f"{DRIVE_SPLITS_DIR}/{s}.txt")
    shutil.copy(f"{LOCAL_SPLITS_DIR}/{s}.jsonl", f"{DRIVE_SPLITS_DIR}/{s}.jsonl")
    print(f"  ✅ Synced {s} splits to Drive")
print("All split files safely backed up to Google Drive!")

## 3. Retrain SentencePiece 32k BPE & Final Benchmark

In [ ]:
LOCAL_MPREFIX = f"{LOCAL_TOKENIZER_DIR}/nepali_bpe_32k"
DRIVE_MPREFIX = f"{DRIVE_TOKENIZER_DIR}/nepali_bpe_32k"
TRAIN_TXT = f"{LOCAL_SPLITS_DIR}/train.txt"
VAL_TXT   = f"{LOCAL_SPLITS_DIR}/val.txt"

mb = os.path.getsize(TRAIN_TXT) / (1024 * 1024)
print(f"Training BPE on {mb:.1f} MB training text...")

spm.SentencePieceTrainer.train(
    input=TRAIN_TXT, model_prefix=LOCAL_MPREFIX, vocab_size=32000,
    model_type="bpe", character_coverage=0.9995, num_threads=2,
    max_sentence_length=8192, input_sentence_size=500_000,
    shuffle_input_sentence=True,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    pad_piece="<pad>", unk_piece="<unk>", bos_piece="<bos>", eos_piece="<eos>",
    split_by_unicode_script=True, split_by_whitespace=True,
    split_digits=True, byte_fallback=True,
)

sp = spm.SentencePieceProcessor()
sp.load(f"{LOCAL_MPREFIX}.model")
print(f"
Vocab size: {sp.get_piece_size():,}")

# Sync tokenizer models to Google Drive
shutil.copy(f"{LOCAL_MPREFIX}.model", f"{DRIVE_MPREFIX}.model")
shutil.copy(f"{LOCAL_MPREFIX}.vocab", f"{DRIVE_MPREFIX}.vocab")
print(f"✅ Tokenizer model & vocab synced to Google Drive: {DRIVE_MPREFIX}.model")

# Evaluate on Validation set
COMBINING_MARKS = set(chr(c) for c in range(0x0901, 0x0904)) | \n                  set(chr(c) for c in range(0x093E, 0x094E)) | {chr(0x0962), chr(0x0963)}

def count_broken_graphemes(tokens):
    broken = 0
    for tok in tokens:
        clean_tok = tok.lstrip(" ")
        if clean_tok and clean_tok[0] in COMBINING_MARKS:
            broken += 1
    return broken

val_sentences = []
with open(VAL_TXT, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line and len(line) > 20: val_sentences.append(line)
        if len(val_sentences) >= 5000: break

total_words, total_tokens, total_chars, total_broken = 0, 0, 0, 0
token_freqs = collections.Counter()

for text in val_sentences:
    words = text.split()
    tokens = sp.encode(text, out_type=str)
    token_ids = sp.encode(text, out_type=int)
    total_words += len(words)
    total_tokens += len(tokens)
    total_chars += len(text)
    total_broken += count_broken_graphemes(tokens)
    token_freqs.update(token_ids)

fertility = total_tokens / max(1, total_words)
chars_per_tok = total_chars / max(1, total_tokens)
broken_rate = (total_broken / max(1, total_tokens)) * 100
probs = [c / total_tokens for c in token_freqs.values()]
shannon_entropy = -sum(p * math.log2(p) for p in probs if p > 0)
renyi_entropy = -math.log2(sum(p**2 for p in probs))

print("
" + "=" * 75)
print("FINAL INTRINSIC TOKENIZER METRICS (Standard BPE 32k)")
print("=" * 75)
print(f"  Fertility:          {fertility:.2f} tokens / word")
print(f"  Characters / Token: {chars_per_tok:.2f}")
print(f"  Broken Matra Rate:  {broken_rate:.3f}%")
print(f"  Shannon Entropy:    {shannon_entropy:.2f} bits")
print(f"  Rényi Entropy:      {renyi_entropy:.2f} bits")
print("=" * 75)